# **Phần 2: Hệ Thống Đánh Giá Đối Chứng Đa Mô Hình Với Kiến Trúc Phân Tán**

Notebook này đóng vai trò là đánh giá thực nghiệm cho đồ án. Sau khi thực hiện Fine-tuning thành công 3 mô hình ngôn ngữ lớn (LLMs), chúng ta triển khai một quy trình đánh giá chuẩn nghiên cứu bao gồm:

* **Sequential GPU Inference:** Chiến thuật nạp và giải phóng VRAM tuần tự giúp vận hành 3 mô hình (Llama-3, Qwen-2.5, DeepSeek-R1) trên cùng một tài nguyên card đồ họa T4 giới hạn của Kaggle.
* **Kiến trúc Big Data (Dask):** Sử dụng cơ chế *Lazy Evaluation* của Dask DataFrame để mô phỏng việc xử lý dữ liệu phân tán. Điều này đảm bảo hệ thống có khả năng mở rộng (scalable) khi quy mô dữ liệu tăng từ hàng trăm lên hàng triệu dòng.
* **Benchmarking:** Đối chiếu trực tiếp với các chỉ số *State-of-the-art* (GPT-4o, FinBERT, BERT) từ bài báo gốc để xác lập giá trị thực tiễn và tính tối ưu chi phí của giải pháp mã nguồn mở.

In [1]:
%%capture
!pip install "unsloth[kaggle-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --upgrade --no-deps trl

import warnings, os
warnings.filterwarnings('ignore')
import transformers
transformers.logging.set_verbosity_error()
os.makedirs('/kaggle/working/dataset', exist_ok=True)

# **Tìm đường dẫn các LoRA Adapter và Tải Test Set**
Tự động tìm adapter của từng mô hình từ output các notebook fine-tuning đã Add Input.

In [2]:
import glob, pandas as pd

def find_adapter(folder_name):
    """Tự động tìm đường dẫn adapter từ Kaggle input."""
    paths = glob.glob(f"/kaggle/input/**/{folder_name}", recursive=True)
    if paths:
        print(f"   Tìm thấy [{folder_name}]: {paths[0]}")
        return paths[0]
    print(f"   Không tìm thấy [{folder_name}]")
    return None

print("Đường dẫn các LoRA Adapter:")
LLAMA_PATH    = find_adapter("lora_model")
QWEN_PATH     = find_adapter("lora_qwen")
DEEPSEEK_PATH = find_adapter("lora_deepseek")

# Tải test set
test_path = glob.glob("/kaggle/input/**/test_set.csv", recursive=True)
test_path = test_path[0] if test_path else "/kaggle/working/dataset/test_set.csv"
test_df   = pd.read_csv(test_path)
print(f"\n Test set: {len(test_df)} mẫu từ {test_path}")

Đường dẫn các LoRA Adapter:
   Tìm thấy [lora_model]: /kaggle/input/notebooks/nhatkhoapm/bigdata-llama-preprocessing-finetunning/lora_model
   Tìm thấy [lora_qwen]: /kaggle/input/notebooks/nhatkhoapm/bigdata-qwen-preprocessing-finetunning/lora_qwen
   Tìm thấy [lora_deepseek]: /kaggle/input/notebooks/nhatkhoapm/bigdata-deepseek-preprocessing-finetunning/lora_deepseek

 Test set: 516 mẫu từ /kaggle/input/notebooks/nhatkhoapm/bigdata-deepseek-preprocessing-finetunning/dataset/test_set.csv


# **Định nghĩa Hàm Dùng Chung cho cả 3 Mô hình**
Hàm `predict_batch` xử lý nhiều câu cùng lúc (batch inference) trên GPU.
Hàm `parse_output` xử lý output JSON — kể cả trường hợp DeepSeek-R1 có thêm
thẻ `<think>...</think>` (reasoning trace) trước câu trả lời thực sự.

In [3]:
import re, json, torch

inference_prompt = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
You are an AI assistant specializing in financial sentiment classification. Your task is to analyze each financial sentence and classify it as negative, positive, or neutral. Provide your final classification in the following JSON format without explanations: {{"Sentiment": "sentiment_tag"}}.

### Input:
{sentence}

### Response:
"""

def parse_output(generated: str) -> str:
    """
    Parse JSON output từ model.
    Xử lý đặc biệt cho DeepSeek-R1: loại bỏ <think>...</think> trước khi parse.
    """
    # Loại bỏ reasoning trace của DeepSeek-R1
    clean = re.sub(r'<think>.*?</think>', '', generated, flags=re.DOTALL).strip()
    try:
        result = json.loads(clean)
        return result.get("Sentiment", "unknown").lower()
    except Exception:
        match = re.search(r'"Sentiment"\s*:\s*"(\w+)"', clean, re.IGNORECASE)
        return match.group(1).lower() if match else "unknown"


def predict_batch(model, tokenizer, sentences: list, batch_size: int = 16) -> list:
    """
    Inference theo batch — nhanh hơn ~10-15x so với từng câu một.
    
    Thực hiện dự đoán hàng loạt để tối ưu GPU.
    Bổ sung max_new_tokens cao hơn để hỗ trợ mô hình suy luận (DeepSeek).
    """
    all_preds = []
    prompts   = [inference_prompt.format(sentence=s) for s in sentences]

    for start in range(0, len(prompts), batch_size):
        batch_prompts = prompts[start : start + batch_size]

        # Đảm bảo padding bên trái để mô hình sinh văn bản chính xác
        tokenizer.padding_side = "left"
        inputs = tokenizer(
            batch_prompts,
            return_tensors = "pt",
            padding        = True,
            truncation     = True,
            max_length     = 512,
        ).to("cuda")

        with torch.no_grad():
            outputs = model.generate(
                input_ids      = inputs["input_ids"],
                attention_mask = inputs["attention_mask"],
                max_new_tokens = 300,      # DeepSeek cần nhiều hơn vì có <think>
                do_sample      = False,
                pad_token_id   = tokenizer.eos_token_id,
                use_cache      = True,
            )

        # Trích xuất phần nội dung mới được sinh ra (bỏ phần prompt gốc)
        input_len = inputs["input_ids"].shape[1]
        for out in outputs:
            generated = tokenizer.decode(out[input_len:], skip_special_tokens=True)
            # Hàm parse_output sẽ dùng Regex để 'vét' JSON trong output
            all_preds.append(parse_output(generated))

    return all_preds


def run_inference(model, tokenizer, test_df, model_name, batch_size=16):
    """Chạy toàn bộ inference và in tiến độ."""
    import time
    print(f"\n [{model_name}] Bắt đầu inference {len(test_df)} câu | batch={batch_size}")
    print("-" * 55)

    start_time    = time.time()
    all_sentences = test_df['Sentence'].tolist()
    predictions   = []

    for start in range(0, len(all_sentences), batch_size):
        batch       = all_sentences[start : start + batch_size]
        batch_preds = predict_batch(model, tokenizer, batch, batch_size=len(batch))
        predictions.extend(batch_preds)

        end_idx = min(start + batch_size, len(all_sentences))
        elapsed = time.time() - start_time
        print(f"  [{end_idx:>3}/{len(all_sentences)}] "
              f"{elapsed:.0f}s | {end_idx/elapsed:.1f} câu/s")

    total_time = time.time() - start_time
    print(f" [{model_name}] Hoàn thành: {total_time:.1f}s (~{total_time/60:.1f} phút)")
    return predictions, total_time

# **Inference — Mô hình 1: Llama-3 8B + LoRA**

In [4]:
import gc
from unsloth import FastLanguageModel
from kaggle_secrets import UserSecretsClient

try:
    hf_token = UserSecretsClient().get_secret("HF_TOKEN")
except:
    hf_token = None

# Load Llama
print("Load Llama-3 8B + LoRA Adapter...")
model_llama, tokenizer_llama = FastLanguageModel.from_pretrained(
    model_name     = LLAMA_PATH,
    max_seq_length = 512,
    dtype          = None,
    load_in_4bit   = True,
    token          = hf_token,
)
FastLanguageModel.for_inference(model_llama)

# Inference
preds_llama, time_llama = run_inference(
    model_llama, tokenizer_llama, test_df, "Llama-3 8B"
)

# Lưu kết quả
df_llama = test_df.copy()
df_llama['Predicted_Sentiment'] = preds_llama
df_llama.to_csv('/kaggle/working/dataset/predictions_llama.csv', index=False)
print("Đã lưu predictions_llama.csv")

# Giải phóng GPU trước khi load model tiếp theo
del model_llama, tokenizer_llama
gc.collect()
torch.cuda.empty_cache()
print("Đã giải phóng GPU memory")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
Load Llama-3 8B + LoRA Adapter...
==((====))==  Unsloth 2026.5.10: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/5.70G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/198 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.25k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/50.6k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/350 [00:00<?, ?B/s]


 [Llama-3 8B] Bắt đầu inference 516 câu | batch=16
-------------------------------------------------------
  [ 16/516] 11s | 1.5 câu/s
  [ 32/516] 15s | 2.1 câu/s
  [ 48/516] 20s | 2.4 câu/s
  [ 64/516] 24s | 2.7 câu/s
  [ 80/516] 28s | 2.8 câu/s
  [ 96/516] 33s | 2.9 câu/s
  [112/516] 37s | 3.0 câu/s
  [128/516] 42s | 3.1 câu/s
  [144/516] 46s | 3.1 câu/s
  [160/516] 51s | 3.1 câu/s
  [176/516] 56s | 3.2 câu/s
  [192/516] 60s | 3.2 câu/s
  [208/516] 65s | 3.2 câu/s
  [224/516] 70s | 3.2 câu/s
  [240/516] 75s | 3.2 câu/s
  [256/516] 80s | 3.2 câu/s
  [272/516] 86s | 3.2 câu/s
  [288/516] 91s | 3.2 câu/s
  [304/516] 97s | 3.1 câu/s
  [320/516] 103s | 3.1 câu/s
  [336/516] 109s | 3.1 câu/s
  [352/516] 115s | 3.1 câu/s
  [368/516] 121s | 3.0 câu/s
  [384/516] 128s | 3.0 câu/s
  [400/516] 134s | 3.0 câu/s
  [416/516] 141s | 2.9 câu/s
  [432/516] 148s | 2.9 câu/s
  [448/516] 155s | 2.9 câu/s
  [464/516] 162s | 2.9 câu/s
  [480/516] 169s | 2.8 câu/s
  [496/516] 176s | 2.8 câu/s
  [512/516] 

# **Inference — Mô hình 2: Qwen2.5 7B + LoRA**

In [5]:
# Load Qwen
print("Load Qwen2.5 7B + LoRA Adapter...")
model_qwen, tokenizer_qwen = FastLanguageModel.from_pretrained(
    model_name     = QWEN_PATH,
    max_seq_length = 512,
    dtype          = None,
    load_in_4bit   = True,
    token          = hf_token,
)
FastLanguageModel.for_inference(model_qwen)

# Inference
preds_qwen, time_qwen = run_inference(
    model_qwen, tokenizer_qwen, test_df, "Qwen2.5 7B"
)

# Lưu kết quả
df_qwen = test_df.copy()
df_qwen['Predicted_Sentiment'] = preds_qwen
df_qwen.to_csv('/kaggle/working/dataset/predictions_qwen.csv', index=False)
print("Đã lưu predictions_qwen.csv")

# Giải phóng GPU
del model_qwen, tokenizer_qwen
gc.collect()
torch.cuda.empty_cache()
print("Đã giải phóng GPU memory")

Load Qwen2.5 7B + LoRA Adapter...
==((====))==  Unsloth 2026.5.10: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/5.55G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/172 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.35k [00:00<?, ?B/s]

...

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/605 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/617 [00:00<?, ?B/s]

unsloth/qwen2.5-7b-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.

 [Qwen2.5 7B] Bắt đầu inference 516 câu | batch=16
-------------------------------------------------------
  [ 16/516] 6s | 2.6 câu/s
  [ 32/516] 12s | 2.7 câu/s
  [ 48/516] 18s | 2.7 câu/s
  [ 64/516] 23s | 2.7 câu/s
  [ 80/516] 29s | 2.7 câu/s
  [ 96/516] 35s | 2.7 câu/s
  [112/516] 42s | 2.7 câu/s
  [128/516] 48s | 2.6 câu/s
  [144/516] 55s | 2.6 câu/s
  [160/516] 61s | 2.6 câu/s
  [176/516] 67s | 2.6 câu/s
  [192/516] 73s | 2.6 câu/s
  [208/516] 79s | 2.6 câu/s
  [224/516] 85s | 2.6 câu/s
  [240/516] 91s | 2.6 câu/s
  [256/516] 96s | 2.7 câu/s
  [272/516] 102s | 2.7 câu/s
  [288/516] 108s | 2.7 câu/s
  [304/516] 114s | 2.7 câu/s
  [320/516] 120s | 2.7 câu/s
  [336/516] 126s | 2.7 câu/s
  [352/516] 133s | 2.7 câu/s
  [368/516] 139s | 2.6 câu/s
  [384/516] 145s | 2.6 câu/s
  [400/516] 151s | 2.6 câu/s
  [416/516] 157s | 2.6 câu/s
  [432/516] 163s | 2.6 câu/s
  [448/516] 169s | 2.6 câu/s
  

# **Inference — Mô hình 3: DeepSeek-R1 Distill Qwen 7B + LoRA**
DeepSeek-R1 là mô hình lý luận (reasoning model) — trước khi trả lời nó sinh ra
một chuỗi suy nghĩ nội tâm trong thẻ `<think>...</think>`. Hàm `parse_output`
đã xử lý tự động để chỉ lấy phần JSON kết quả cuối cùng.

In [6]:
# Load DeepSeek
print("Load DeepSeek-R1 Distill Qwen 7B + LoRA Adapter...")
model_deepseek, tokenizer_deepseek = FastLanguageModel.from_pretrained(
    model_name     = DEEPSEEK_PATH,
    max_seq_length = 512,
    dtype          = None,
    load_in_4bit   = True,
    token          = hf_token,
)
FastLanguageModel.for_inference(model_deepseek)

# Inference
preds_deepseek, time_deepseek = run_inference(
    model_deepseek, tokenizer_deepseek, test_df, "DeepSeek-R1 7B"
)

# Lưu kết quả
df_deepseek = test_df.copy()
df_deepseek['Predicted_Sentiment'] = preds_deepseek
df_deepseek.to_csv('/kaggle/working/dataset/predictions_deepseek.csv', index=False)
print("Đã lưu predictions_deepseek.csv")

# Giải phóng GPU
del model_deepseek, tokenizer_deepseek
gc.collect()
torch.cuda.empty_cache()
print("Đã giải phóng GPU memory")

Load DeepSeek-R1 Distill Qwen 7B + LoRA Adapter...
==((====))==  Unsloth 2026.5.10: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/5.55G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/236 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.40k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/6.78k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/472 [00:00<?, ?B/s]

unsloth/DeepSeek-R1-Distill-Qwen-7B-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.

 [DeepSeek-R1 7B] Bắt đầu inference 516 câu | batch=16
-------------------------------------------------------
  [ 16/516] 6s | 2.6 câu/s
  [ 32/516] 12s | 2.6 câu/s
  [ 48/516] 18s | 2.7 câu/s
  [ 64/516] 24s | 2.7 câu/s
  [ 80/516] 30s | 2.7 câu/s
  [ 96/516] 37s | 2.6 câu/s
  [112/516] 43s | 2.6 câu/s
  [128/516] 50s | 2.6 câu/s
  [144/516] 56s | 2.6 câu/s
  [160/516] 62s | 2.6 câu/s
  [176/516] 68s | 2.6 câu/s
  [192/516] 74s | 2.6 câu/s
  [208/516] 80s | 2.6 câu/s
  [224/516] 85s | 2.6 câu/s
  [240/516] 91s | 2.6 câu/s
  [256/516] 97s | 2.6 câu/s
  [272/516] 102s | 2.7 câu/s
  [288/516] 109s | 2.7 câu/s
  [304/516] 114s | 2.7 câu/s
  [320/516] 120s | 2.7 câu/s
  [336/516] 127s | 2.6 câu/s
  [352/516] 134s | 2.6 câu/s
  [368/516] 140s | 2.6 câu/s
  [384/516] 146s | 2.6 câu/s
  [400/516] 152s | 2.6 câu/s
  [416/516] 158s | 2.6 câu/s
  [432/516] 164s | 2.6 câu/s
  [448/516

# **Phân tích Phân tán với Dask — Cả 3 Mô hình**
Dùng Dask để tính Accuracy phân tán cho từng mô hình và in Confusion Matrix.

In [7]:
import dask.dataframe as dd
from dask.diagnostics import ProgressBar

MODEL_FILES = {
    "Llama-3 8B"     : "/kaggle/working/dataset/predictions_llama.csv",
    "Qwen2.5 7B"     : "/kaggle/working/dataset/predictions_qwen.csv",
    "DeepSeek-R1 7B" : "/kaggle/working/dataset/predictions_deepseek.csv",
}

print("Đang phân tích phân tán bằng Dask...\n")

dask_results = {}

for model_name, csv_path in MODEL_FILES.items():
    print(f"{'='*55}")
    print(f"  {model_name}")
    print(f"{'='*55}")

    ddf = dd.read_csv(csv_path)
    ddf = ddf.assign(
        actual    = ddf['Sentiment'].astype(str).str.lower().str.strip(),
        predicted = ddf['Predicted_Sentiment'].astype(str).str.lower().str.strip()
    )

    valid_labels = ['positive', 'negative', 'neutral']
    ddf_valid    = ddf[ddf['predicted'].isin(valid_labels)]

    with ProgressBar():
        # Mẫu số phải là TOÀN BỘ tập dữ liệu (516 câu)
        # Không được dùng ddf_valid để tính tổng, vì 'unknown' cũng là một lỗi dự đoán.
        total_count = len(ddf)
        
        # Đếm số lượng dự đoán khớp hoàn toàn với nhãn thực tế
        correct_count = (ddf['actual'] == ddf['predicted']).sum().compute()
        
        # Tính toán Accuracy thực tế (phạt cả những câu đoán ra 'unknown')
        accuracy_final = (correct_count / total_count) * 100

        conf_mat = (
            ddf_valid.groupby(['actual', 'predicted'])
            .size().compute().reset_index(name='count')
        )

    print(f"  Tổng mẫu kiểm thử : {total_count}")
    print(f"  Dự đoán đúng      : {correct_count}")
    print(f"  Accuracy          : {accuracy_final:.2f}%")
    print(f"\n  Confusion Matrix:")
    print(conf_mat.to_string(index=False))
    print("\n")

    dask_results[model_name] = {
        'total': total_count, 
        'correct': correct_count, 
        'accuracy': accuracy_final
    }

print(f"\nDask phân tích phân tán hoàn thành!")

Đang phân tích phân tán bằng Dask...

  Llama-3 8B
[########################################] | 100% Completed | 102.16 ms
[########################################] | 100% Completed | 102.38 ms
[########################################] | 100% Completed | 102.49 ms
  Tổng mẫu kiểm thử : 516
  Dự đoán đúng      : 449
  Accuracy          : 87.02%

  Confusion Matrix:
  actual predicted  count
negative  negative    154
negative   neutral     10
negative  positive      8
 neutral  negative     23
 neutral   neutral    142
 neutral  positive      7
positive  negative      6
positive   neutral     13
positive  positive    153


  Qwen2.5 7B
[########################################] | 100% Completed | 102.56 ms
[########################################] | 100% Completed | 102.65 ms
[########################################] | 100% Completed | 102.52 ms
  Tổng mẫu kiểm thử : 516
  Dự đoán đúng      : 438
  Accuracy          : 84.88%

  Confusion Matrix:
  actual predicted  count
negative  ne

# **Đánh giá Học thuật và So sánh Hiệu năng**

Trong phần này, chúng ta sử dụng thư viện `scikit-learn` để tính toán các chỉ số thống kê quan trọng (Accuracy, F1-Score, Precision, Recall). Điểm khác biệt cốt lõi trong logic xử lý của nhóm so với các đoạn code cơ bản là:

**Accuracy = (True Positives + True Negatives) / Total Test Samples**

Trong đó, *Total Test Samples* được giữ nguyên tuyệt đối là **516 mẫu**. Nếu mô hình sinh ra lỗi định dạng (ví dụ: sinh ra văn bản rác nên hàm regex trả về `unknown`), mẫu đó sẽ tự động bị tính là một lần dự đoán sai và bị phạt vào tổng điểm. 

Cách tiếp cận này giúp loại bỏ triệt để hiện tượng "Thiên kiến sống sót" (Survival Bias) - tức là chỉ tính điểm trên những câu mô hình đoán ra form chuẩn. Nhờ vậy, kết quả báo cáo của nhóm là trung thực, khắt khe và phản ánh đúng 100% năng lực suy luận thực tế của từng mô hình AI.

In [8]:
import pandas as pd
from sklearn.metrics import (accuracy_score, f1_score,
                              precision_score, recall_score, classification_report)

MODEL_FILES = {
    "Llama-3 8B"     : "/kaggle/working/dataset/predictions_llama.csv",
    "Qwen2.5 7B"     : "/kaggle/working/dataset/predictions_qwen.csv",
    "DeepSeek-R1 7B" : "/kaggle/working/dataset/predictions_deepseek.csv",
}
MODEL_TIMES = {
    "Llama-3 8B"     : time_llama,
    "Qwen2.5 7B"     : time_qwen,
    "DeepSeek-R1 7B" : time_deepseek,
}

our_results = {}

for model_name, csv_path in MODEL_FILES.items():
    df = pd.read_csv(csv_path)
    df['actual']    = df['Sentiment'].astype(str).str.lower().str.strip()
    df['predicted'] = df['Predicted_Sentiment'].astype(str).str.lower().str.strip()

    # Đếm số lượng mẫu hợp lệ và không hợp lệ (mô hình sinh ra rác/unknown)
    valid_mask = df['predicted'].isin(['positive', 'negative', 'neutral'])
    invalid = len(df) - valid_mask.sum()
    
    # Dùng toàn bộ tập test (df) để tính điểm, phạt nặng những câu đoán sai định dạng
    y_true, y_pred = df['actual'], df['predicted']

    # Tính toán các chỉ số (bổ sung zero_division=0 để tránh cảnh báo thư viện khi gặp nhãn unknown)
    acc  = accuracy_score(y_true, y_pred)
    f1   = f1_score(y_true, y_pred, average='weighted')
    prec = precision_score(y_true, y_pred, average='weighted', zero_division=0)
    rec  = recall_score(y_true, y_pred, average='weighted', zero_division=0)

    our_results[model_name] = {
        'accuracy': acc, 'f1': f1, 'precision': prec, 'recall': rec,
        'invalid': invalid, 'time': MODEL_TIMES[model_name]
    }

    print(f"\n{'='*60}")
    print(f"  {model_name}")
    print(f"{'='*60}")
    print(f"  Hợp lệ   : {valid_mask.sum()}/{len(df)} | Không hợp lệ: {invalid}")
    print(f"  Accuracy : {acc:.4f} ({acc*100:.2f}%)")
    print(f"  F1-Score : {f1:.4f}")
    print(f"  Precision: {prec:.4f}")
    print(f"  Recall   : {rec:.4f}")
    print(f"  Thời gian: {MODEL_TIMES[model_name]:.1f}s (~{MODEL_TIMES[model_name]/60:.1f} phút)")
    print(f"\n  Chi tiết từng class:")
    
    # Thiết lập labels explicitly để loại bỏ các nhãn rác khỏi bảng in chi tiết, 
    # nhưng số liệu tổng Accuracy ở trên vẫn đã bị trừ điểm
    print(classification_report(y_true, y_pred,
          labels=['negative', 'neutral', 'positive'],
          target_names=['negative', 'neutral', 'positive'],
          zero_division=0))

# Bảng so sánh tổng hợp
r = our_results
comparison_df = pd.DataFrame({
    'Model': [
        'ft:gpt-4o (Paper)',
        'ft:gpt-4o-mini (Paper)',
        'ft:bert (Paper)',
        'ft:finbert (Paper)',
        't:svm (Paper)',
        't:random-forest (Paper)',
        't:logistic-regression (Paper)',
        '★ Llama-3 8B + LoRA (Nhóm)',
        '★ Qwen2.5 7B + LoRA (Nhóm)',
        '★ DeepSeek-R1 7B + LoRA (Nhóm)',
    ],
    'Accuracy': [
        0.8779, 0.8779, 0.8120, 0.8004, 0.6453, 0.6531, 0.6492,
        round(r['Llama-3 8B']['accuracy'],    4),
        round(r['Qwen2.5 7B']['accuracy'],    4),
        round(r['DeepSeek-R1 7B']['accuracy'],4),
    ],
    'F1-Score': [
        0.8769, 0.8773, 0.8116, 0.8004, 0.6438, 0.6531, 0.6474,
        round(r['Llama-3 8B']['f1'],    4),
        round(r['Qwen2.5 7B']['f1'],    4),
        round(r['DeepSeek-R1 7B']['f1'],4),
    ],
    'Chi phí': [
        '$13.08', '$1.57', 'Free', 'Free', 'Free', 'Free', 'Free',
        'Free (Kaggle GPU)',
        'Free (Kaggle GPU)',
        'Free (Kaggle GPU)',
    ],
}).sort_values('Accuracy', ascending=False).reset_index(drop=True)

print("\n" + "="*75)
print("        BẢNG SO SÁNH TỔNG HỢP VỚI BÀI BÁO GỐC")
print("="*75)
print(comparison_df.to_string(index=False))
print("="*75)

# Kết luận
print("\nKẾT LUẬN NGHIÊN CỨU:")
for name, res in our_results.items():
    gap = 0.8779 - res['accuracy']
    print(f"\n  [{name}] Accuracy = {res['accuracy']*100:.2f}% | "
          f"Chênh lệch vs GPT-4o-mini: {gap:+.4f}")
    if res['accuracy'] >= 0.80:
        print(f"    Đạt ngang BERT-class với chi phí MIỄN PHÍ!")
    elif res['accuracy'] >= 0.70:
        print(f"    Vượt trội SVM/RF/LR với chi phí MIỄN PHÍ!")
    else:
        print(f"     Cần tăng max_steps để cải thiện thêm.")


  Llama-3 8B
  Hợp lệ   : 516/516 | Không hợp lệ: 0
  Accuracy : 0.8702 (87.02%)
  F1-Score : 0.8701
  Precision: 0.8710
  Recall   : 0.8702
  Thời gian: 185.2s (~3.1 phút)

  Chi tiết từng class:
              precision    recall  f1-score   support

    negative       0.84      0.90      0.87       172
     neutral       0.86      0.83      0.84       172
    positive       0.91      0.89      0.90       172

    accuracy                           0.87       516
   macro avg       0.87      0.87      0.87       516
weighted avg       0.87      0.87      0.87       516


  Qwen2.5 7B
  Hợp lệ   : 516/516 | Không hợp lệ: 0
  Accuracy : 0.8488 (84.88%)
  F1-Score : 0.8483
  Precision: 0.8491
  Recall   : 0.8488
  Thời gian: 195.8s (~3.3 phút)

  Chi tiết từng class:
              precision    recall  f1-score   support

    negative       0.84      0.90      0.87       172
     neutral       0.84      0.79      0.81       172
    positive       0.87      0.85      0.86       172

    a

In [9]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from math import pi
from sklearn.metrics import confusion_matrix

# 1. Tạo thư mục chứa ảnh
os.makedirs('figures', exist_ok=True)
print("Đang khởi tạo vẽ biểu đồ tự động từ kết quả Evaluation...")

# ---------------------------------------------------------
# 2. Tạo 3 ảnh Confusion Matrix tự động (Hình cho Llama, Qwen, DeepSeek)
# ---------------------------------------------------------
labels = ['negative', 'neutral', 'positive']
color_maps = {'Llama-3 8B': 'Blues', 'Qwen2.5 7B': 'Greens', 'DeepSeek-R1 7B': 'Reds'}
cms = {}

# Tự động đọc file và tính Ma trận nhầm lẫn bằng Sklearn
for model_name, csv_path in MODEL_FILES.items():
    df_temp = pd.read_csv(csv_path)
    y_true = df_temp['Sentiment'].astype(str).str.lower().str.strip()
    y_pred = df_temp['Predicted_Sentiment'].astype(str).str.lower().str.strip()
    
    # Ép khung 3x3, tự động bỏ qua các nhãn rác (unknown) để không làm vỡ hình
    matrix = confusion_matrix(y_true, y_pred, labels=labels)
    cms[model_name] = (matrix, color_maps[model_name])

for name, (matrix, cmap) in cms.items():
    plt.figure(figsize=(6, 5))
    sns.heatmap(matrix, annot=True, fmt='d', cmap=cmap,
                xticklabels=labels, yticklabels=labels, cbar=False, annot_kws={"size": 14})
    plt.title(f'Confusion Matrix - {name}')
    plt.xlabel('Predicted Label')
    plt.ylabel('Actual Label')
    
    # Xử lý tên file an toàn (xóa dấu cách, dấu chấm)
    safe_name = name.replace(" ", "_").replace(".", "").replace("-", "_")
    plt.savefig(f'figures/confusion_matrix_{safe_name}.png', bbox_inches='tight', dpi=150)
    plt.close()
print("✅ Đã tạo xong 3 ảnh Confusion Matrix động.")

# ---------------------------------------------------------
# 3. Ảnh Performance Comparison (Biểu đồ ngang so sánh Accuracy)
# ---------------------------------------------------------
# Rút trích Accuracy tự động từ biến our_results của cell Sklearn bên trên
acc_llama    = our_results['Llama-3 8B']['accuracy'] * 100
acc_qwen     = our_results['Qwen2.5 7B']['accuracy'] * 100
acc_deepseek = our_results['DeepSeek-R1 7B']['accuracy'] * 100

model_names = ['GPT-4o (Bài báo)', 'GPT-4o-mini', 'Llama-3 (Nhóm)', 'DeepSeek-R1 (Nhóm)', 'Qwen-2.5 (Nhóm)', 'FinBERT']
acc_scores = [87.79, 87.79, acc_llama, acc_deepseek, acc_qwen, 80.04]

plt.figure(figsize=(9, 5))
bars = plt.barh(model_names[::-1], acc_scores[::-1], color=sns.color_palette("magma", len(model_names)))
plt.xlabel('Accuracy (%)')
plt.title('So Sánh Độ Chính Xác (Accuracy) Giữa Các Mô Hình')
plt.xlim(75, 90) # Phóng to khoảng 75-90 để thấy rõ chênh lệch

for bar in bars:
    plt.text(bar.get_width() + 0.2, bar.get_y() + bar.get_height()/2,
             f'{bar.get_width():.2f}%', va='center', ha='left', fontweight='bold')
plt.savefig('figures/performance_comparison.png', bbox_inches='tight', dpi=150)
plt.close()
print("✅ Đã tạo xong ảnh Performance Comparison động.")

# ---------------------------------------------------------
# 4. Ảnh Radar Chart (Đánh giá đa chiều TÍNH TOÁN TỰ ĐỘNG)
# ---------------------------------------------------------
categories = ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'Efficiency']
N = len(categories)

# --- THUẬT TOÁN CHUẨN HÓA (NORMALIZATION) ---
# Tự động tìm thời gian chạy nhanh nhất trong 3 model (Best Time)
min_time = min([res['time'] for res in our_results.values()])

# Hàm lấy thông số tự động từ biến our_results
def get_radar_values(model_name):
    res = our_results[model_name]
    
    # Tính điểm Efficiency tự động dựa trên thời gian chạy (time)
    # Model nào chạy nhanh bằng Min_Time sẽ được điểm trần (88.0 để cân xứng với Accuracy)
    # Model nào chạy chậm hơn sẽ bị hạ điểm theo tỷ lệ
    efficiency_score = (min_time / res['time']) * 88.0 
    
    return [res['accuracy']*100, res['precision']*100, res['recall']*100, res['f1']*100, efficiency_score]

# Lấy dữ liệu đã được tính toán tự động
values_llama    = get_radar_values('Llama-3 8B')
values_qwen     = get_radar_values('Qwen2.5 7B')
values_deepseek = get_radar_values('DeepSeek-R1 7B')

angles = [n / float(N) * 2 * pi for n in range(N)]
angles += angles[:1]
values_llama += values_llama[:1]
values_qwen += values_qwen[:1]
values_deepseek += values_deepseek[:1]

plt.figure(figsize=(6, 6))
ax = plt.subplot(111, polar=True)
plt.xticks(angles[:-1], categories, size=10)

ax.plot(angles, values_llama, linewidth=2, linestyle='solid', label='Llama-3 8B', color='#1f77b4')
ax.fill(angles, values_llama, '#1f77b4', alpha=0.1)

ax.plot(angles, values_qwen, linewidth=2, linestyle='solid', label='Qwen-2.5 7B', color='#2ca02c')
ax.fill(angles, values_qwen, '#2ca02c', alpha=0.1)

ax.plot(angles, values_deepseek, linewidth=2, linestyle='solid', label='DeepSeek-R1 7B', color='#d62728')
ax.fill(angles, values_deepseek, '#d62728', alpha=0.1)

plt.title('Radar Chart: Đánh giá đa chiều các LLMs', size=14, y=1.1, fontweight='bold')
plt.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1))
plt.savefig('figures/radar_chart.png', bbox_inches='tight', dpi=150)
plt.close()
print("✅ Đã tạo xong ảnh Radar Chart động (Efficiency đã được chuẩn hóa theo Time).")

# ---------------------------------------------------------
# 5. Bảng SOTA Comparison (Biến DataFrame thành ảnh Table)
# ---------------------------------------------------------
sota_df = pd.DataFrame({
    'Model': ['GPT-4o', 'GPT-4o-mini', 'Llama-3 (Nhóm)', 'DeepSeek-R1 (Nhóm)', 'Qwen-2.5 (Nhóm)', 'FinBERT'],
    'Type': ['Closed-API', 'Closed-API', 'Open-Source', 'Open-Source', 'Open-Source', 'Open-Source'],
    'Accuracy (%)': [87.79, 87.79, round(acc_llama, 2), round(acc_deepseek, 2), round(acc_qwen, 2), 80.04],
    'Cost / Run': ['$13.08', '$1.57', '$0', '$0', '$0', '$0']
})

fig, ax = plt.subplots(figsize=(9, 3))
ax.axis('off')
table = ax.table(cellText=sota_df.values, colLabels=sota_df.columns, cellLoc='center', loc='center')
table.auto_set_font_size(False)
table.set_fontsize(11)
table.scale(1.2, 1.8)
plt.title('Bảng So Sánh Với Các Mô Hình SOTA (State of the Art)', fontweight='bold', pad=20)
plt.savefig('figures/sota_comparison.png', bbox_inches='tight', dpi=150)
plt.close()
print("✅ Đã tạo xong ảnh SOTA Comparison Table động.")
print("🎉 HOÀN TẤT! Toàn bộ ảnh đã được tự động cập nhật số liệu và lưu trong mục figures.")

Đang khởi tạo vẽ biểu đồ tự động từ kết quả Evaluation...
✅ Đã tạo xong 3 ảnh Confusion Matrix động.
✅ Đã tạo xong ảnh Performance Comparison động.
✅ Đã tạo xong ảnh Radar Chart động (Efficiency đã được chuẩn hóa theo Time).
✅ Đã tạo xong ảnh SOTA Comparison Table động.
🎉 HOÀN TẤT! Toàn bộ ảnh đã được tự động cập nhật số liệu và lưu trong mục figures.
